# Spyglass Setup & First Ingestion

This notebook walks through:
1. Connecting to your MySQL database
2. Configuring Spyglass
3. Ingesting your first NWB file
4. Querying the Common tables
5. Reading your behavioural trial data back out

**Prerequisites:**
- `conda activate spyglass`
- MySQL Docker container running (`docker ps` should show `spyglass-db`)
- At least one `.nwb` file in `~/spyglass_data/raw/`

## 0. Check Docker is running

Run this in a terminal before starting:
```bash
docker ps
```
You should see `spyglass-db` in the list. If not:
```bash
docker start spyglass-db
```

## 1. Configure Spyglass

Run this cell **once** to write your DataJoint config file.
After the first run you can skip this cell — the config is saved globally.

In [3]:
import sys
# Remove ~/.local from the path so conda env takes priority
sys.path = [p for p in sys.path if '.local' not in p]

In [4]:
import datajoint as dj
from spyglass.settings import SpyglassConfig

SpyglassConfig().save_dj_config(
    save_method="global",
    base_dir="/home/mpagan/spyglass_data/",
    database_user="root",
    database_password="catt0blepa",  # replace with your actual password
    database_host="localhost",
    database_port=3306,
    set_password=False,
)

print("Config saved.")

ModuleNotFoundError: No module named 'pkg_resources'

## 2. Connect and verify

In [ ]:
import datajoint as dj

# Test the connection
conn = dj.conn()
print(f"Connected: {conn}")

In [ ]:
# Import Spyglass common tables
# This will create all the tables in MySQL on first run — takes ~30 seconds
import spyglass.common as sgc

# Check the Nwbfile table — should be empty at this point
sgc.Nwbfile()

## 3. Copy your NWB file into the raw directory

Spyglass requires NWB files to be in its `raw/` directory.
Do this in a terminal:

```bash
cp /path/to/your/A067_20231105.nwb ~/spyglass_data/raw/
```

Then set the filename below and run the next cell.

In [ ]:
import os

# Set this to your actual NWB filename
NWB_FILENAME = "A067_20231105.nwb"  # <-- change this

raw_dir = "/home/mpagan/spyglass_data/raw/"
full_path = os.path.join(raw_dir, NWB_FILENAME)

assert os.path.exists(full_path), f"File not found: {full_path}"
print(f"Found: {full_path}")

## 4. Ingest the NWB file into Spyglass

In [ ]:
from spyglass.data_import import insert_sessions

# This reads the NWB file and populates all Common tables
# It will print what it finds and any errors
# The first time is slower; subsequent runs skip already-ingested data
insert_sessions(NWB_FILENAME)

## 5. Query the Common tables

After ingestion, the Common tables should have entries for your session.

In [ ]:
# Top-level session info
sgc.Session()

In [ ]:
# Subject/animal info
sgc.Subject()

In [ ]:
# Time intervals registered in this session
sgc.IntervalList()

In [ ]:
# Task epochs
sgc.TaskEpochs()

## 6. Read your trial data back out via PyNWB

Spyglass ingests the session metadata but your custom behavioural columns
(trial_type, outcome, reaction_time, etc.) live in the NWB trials table.
Read them directly via PyNWB — this is the standard approach until you
build a custom Spyglass behaviour pipeline.

In [ ]:
import pynwb
import pandas as pd

nwb_path = f"/home/mpagan/spyglass_data/raw/{NWB_FILENAME}"

with pynwb.NWBHDF5IO(nwb_path, "r") as io:
    nwb = io.read()
    trials = nwb.trials.to_dataframe()

print(f"Loaded {len(trials)} trials")
trials.head(10)

In [ ]:
# Quick summary stats
print("Columns:", list(trials.columns))
print("\nOutcome counts:")
print(trials['outcome'].value_counts())
print("\nTrial type counts:")
print(trials['trial_type'].value_counts())

In [ ]:
import matplotlib.pyplot as plt

# Simple performance plot
hits = trials[trials['outcome'] == 'hit']
errors = trials[trials['outcome'] == 'error']

# Rolling performance
trials['is_hit'] = (trials['outcome'] == 'hit').astype(float)
trials['rolling_perf'] = trials['is_hit'].rolling(window=20, min_periods=5).mean()

fig, axes = plt.subplots(2, 1, figsize=(12, 6))

# Performance over time
axes[0].plot(trials.index, trials['rolling_perf'], color='steelblue')
axes[0].axhline(0.5, color='gray', linestyle='--', alpha=0.5)
axes[0].set_ylabel('Hit rate (rolling 20)')
axes[0].set_title(f'{NWB_FILENAME} — Session performance')

# Reaction time distribution
rt = trials[trials['reaction_time_s'].notna()]['reaction_time_s']
axes[1].hist(rt, bins=50, color='steelblue', edgecolor='white')
axes[1].set_xlabel('Reaction time (s)')
axes[1].set_ylabel('Count')

plt.tight_layout()
plt.show()

## 7. Query across sessions using DataJoint

Once you have multiple sessions ingested, you can query across them.
This is the power of DataJoint — no file loading loops needed.

In [ ]:
# List all ingested sessions
sgc.Session()

In [ ]:
# Filter by subject
subject_id = "A067"  # change to your rat ID
(sgc.Session & {"subject_id": subject_id})

In [ ]:
# Fetch session metadata as a dataframe
sessions_df = pd.DataFrame(
    (sgc.Session & {"subject_id": subject_id}).fetch(as_dict=True)
)
sessions_df

## 8. What's next

If everything above worked, you have a functioning Spyglass installation with
your behavioural data ingested. The next steps are:

1. **Ingest all your sessions** — write a script that loops over your NWB files
   and calls `insert_sessions()` on each one.

2. **Build a custom behaviour pipeline** — create DataJoint tables for your
   standard analyses (psychometric curves, RT distributions, block structure,
   etc.) following the Data → Parameter → Selection → Compute pattern.

3. **Share with the consortium** — once your behaviour pipeline is stable,
   it becomes the standard across all labs using Spyglass for Pro/Anti data.

See `spyglass_conventions.md` for the detailed Frank Lab conventions to follow.